# REDUCCIÓN DE LA DIMENSIONALIDAD

Este notebook trata de reducir el número de variables en nuestro estudio para poder aplicar los distintos modelos posteriores. Transformamos nuestras variables explicativas correlacionadas en un conjunto de componentes ortogonales e independientes. Actuarialmente, esto nos permite sintetizar la información redundante del asegurado en "Perfiles de Riesgo" puros, garantizando que nuestros modelos de reservas posteriores (GLM y BNN) sean matemáticamente estables, no sufran de multicolinealidad y generen estimaciones de incertidumbre fiables para Solvencia II.

En el paso anterior hemos visto que la correlación entre variables numéricas era muy baja con alguna excepción aislada. Esto nos garantiza que nuestros modelos no caerían en multicolinealidad en el caso de usar solamente estas variables. Sin embargo, las variables categóricas contienen información de gran relevancia que será necesaria para obtener la mejor de las estimaciones. 

Debemos comprobar si tiene sentido realizar esta reducción de la dimensionalidad. Para ello y para la futura reducción de la dimensionalidad hay algunas transformaciones de variables que debemos hacer: estandarizar las variables numéricas o reescalar de 0 a 1; codificar las variables categoricas nominales y ordnarias ( comprobar si son nominales algunas como el tipo de auto -> Lujo = Más caro).

### 1. - Transformación de variables numéricas

Los algoritmos que vamos a usar para reducir la sensibilidad son muy sensibles a la escala de nuestros datos, es por ello que decidimos estandarizar las variables numéricas para evitar que los valores mayores de `Customer Lifetime Value` dominen sobre los valores menores que toman otras variables como `Number of Open Complaints`.

Se opta por la estandarización por encima de otros métodos como un reescalado min-max 0-1 para no interferir de una manera artificial sobre la varianza de nuestras variables.

Dentro de las variables predictoras numéricas encontramos dos, que según hemos visto en la exploración de datos, tienen una cola muy pesada a la derecha: `Customer Lifetime Value` y `Monthly Premium Auto`. Antes de estandarizar estas variables debemos transformarlas para asemejarla lo máximo posible a una campana de Gauss. Si usaramos la estandarización directamente sobre estas variables el valor de la media se vería muy distorsionado por los valores gigantes que toma. Para evitar esta situación, optamos por una transformación logarítmica. Esta tansformación comprime los valores que toman estas variables de cola larga, que si bien son valores de interés en el ámbito actuarial al ser extremos, son valores de variables predictoras y no de la variable dependiente. Es por ello que tenemos una mayor libertad de hacer este tratamiento, por último una justificación práctica de la transformación que será necesaria para realizar correctamente la posterior segmentacuón, sería pensar en como nuestro modelo ve a los "clientes", el algoritmo clasificaría a cada uno de los "clientes gigantes" en grupos independientes y dejaría un grupo con la gran mayoría de clientes que toman valores pequeños o medianos enestas variables. Al hacer la transformación logarítmica esperamos clasificar a todos los clientes masivos en el mismo grupo si corresponde o en grupos con otros clientes que correspondan.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler


In [12]:
df = pd.read_csv('../data/processed/df_limpio.csv')
num_cols = ['Customer Lifetime Value', 'Income', 'Monthly Premium Auto', 
            'Months Since Last Claim', 'Months Since Policy Inception', 
            'Number of Open Complaints', 'Number of Policies']


X_num = df[num_cols].copy()

cols_asim = ['Customer Lifetime Value', 'Monthly Premium Auto']

# Creamos las variables transformadas por la transformación logarítmica
for col in cols_asim:
    X_num[f'{col}_log'] = np.log1p(X_num[col])

X_num = X_num.drop(columns = cols_asim)

# Hacemos la estandarización de las variables
fnum_cols = X_num.columns
scaler = StandardScaler()
X_num_scaled = pd.DataFrame(scaler.fit_transform(X_num), columns=fnum_cols)

# Mostramos algunos estadísticos de las variables estandarizadas
print("--- ESTADÍSTICAS TRAS TRANSFORMACIÓN LOGARÍTMICA Y ESTANDARIZACIÓN ---")


sum = 0
for col in fnum_cols:
    print('_____________________________________________________________')
    print(f'Variable: {col.upper()}') 
    
    media = X_num_scaled[col].mean()
    desv  = X_num_scaled[col].std()
    minimo = X_num_scaled[col].min()
    maximo = X_num_scaled[col].max()

    print(f'Media: {media:.4f},    sd: {desv:.4f},    Rango: ({minimo:.4f}, {maximo:.4f})')



--- ESTADÍSTICAS TRAS TRANSFORMACIÓN LOGARÍTMICA Y ESTANDARIZACIÓN ---
_____________________________________________________________
Variable: INCOME
Media: -0.0000,    sd: 1.0001,    Rango: (-1.2319, 2.0578)
_____________________________________________________________
Variable: MONTHS SINCE LAST CLAIM
Media: -0.0000,    sd: 1.0001,    Rango: (-1.4998, 1.9759)
_____________________________________________________________
Variable: MONTHS SINCE POLICY INCEPTION
Media: 0.0000,    sd: 1.0001,    Rango: (-1.7226, 1.8256)
_____________________________________________________________
Variable: NUMBER OF OPEN COMPLAINTS
Media: 0.0000,    sd: 1.0001,    Rango: (-0.4230, 5.0684)
_____________________________________________________________
Variable: NUMBER OF POLICIES
Media: 0.0000,    sd: 1.0001,    Rango: (-0.8236, 2.5207)
_____________________________________________________________
Variable: CUSTOMER LIFETIME VALUE_LOG
Media: -0.0000,    sd: 1.0001,    Rango: (-1.8393, 3.9510)
____________

### 2. - Transformación de Variables categóricas

### 3. - Justificación PCA

### 4. - PCA